# Healthcare innovation sector：Forecast-to-Policy A0--A9 消融实验修正版

本 notebook 专门修正你当前科技/Healthcare innovation sector 消融实验中“读取不到预测数据”的问题。

关键修正如下：

1. 预测消融结果读取目录改为：`D:/paper/onging/ablation_tech`；
2. 按图片中的命名规则读取：`A0_Full_MGT_seed1_pred_raw.csv`、`A0_Full_MGT_seed1_true_raw.csv`、`A1_Transformer_only_seed1_pred_raw.csv` 等；
3. 优先读取 `*_pred_raw.csv`，不再错误读取归一化的 `*_pred.csv` 后重复反归一化；
4. 自动识别 CSV 中的 `date` 列，自动确定交易测试区间，避免 2022 与 2023--2024 区间不一致导致报错；
5. 自动生成与 bank 组一致的 A0--A9 FinRL 长表 CSV、daily return、summary、paper table、best-agent table 和 pivot 表。

运行顺序：从上到下执行即可。若只想先生成与 bank 组当前汇总一致的客观基准表，可在参数单元中设置 `RUN_RL_AGENTS = False`。


In [1]:
# ===============================
# 0. Basic imports and global paths
# ===============================

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ==============================================================
# FinRL 0.3.4 compatibility imports
# ==============================================================
try:
    from finrl.preprocessing.preprocessors import FeatureEngineer
    print("FeatureEngineer import: finrl.preprocessing.preprocessors")
except Exception as e:
    raise ImportError(
        "当前环境无法导入 finrl.preprocessing.preprocessors.FeatureEngineer。\n"
        "请确认 Jupyter kernel 是包含 finrl==0.3.4 的环境。"
    ) from e

try:
    from finrl.preprocessing.preprocessors import data_split as _finrl_data_split
    print("data_split import: finrl.preprocessing.preprocessors")
except Exception:
    _finrl_data_split = None
    print("data_split import: custom fallback data_split")

try:
    from finrl.model.models import DRLAgent
    print("DRLAgent import: finrl.model.models")
except Exception as e:
    raise ImportError(
        "当前环境无法导入 finrl.model.models.DRLAgent。\n"
        "这通常说明 Jupyter kernel 不是你的 FinRL 环境，或 FinRL 安装不完整。"
    ) from e

# Gym / Stable-Baselines3 imports
import gym
from gym import spaces
from gym.utils import seeding
from stable_baselines3.common.vec_env import DummyVecEnv


def _first_existing_path(candidates, description):
    """Return the first existing path; if none exists, return the first candidate and print warning."""
    candidates = [Path(p) for p in candidates]
    for p in candidates:
        if p.exists():
            print(f"[OK] {description}: {p}")
            return p
    print(f"[WARN] 未找到 {description}，先使用默认路径: {candidates[0]}")
    return candidates[0]

# -------------------------------
# Project paths
# -------------------------------
PROJECT_ROOT = Path(r"D:/paper/onging")

# 关键修正：预测消融结果目录，不再读取 D:/paper/onging/ablation
PRED_ABLATION_DIR = PROJECT_ROOT / "ablation_tech"

# 输出目录放在 ablation_tech 下，避免覆盖 bank 组
GENERATED_FINRL_DIR = PRED_ABLATION_DIR / "finrl_csv"
RL_ABLATION_DIR = PRED_ABLATION_DIR / "rl"

PRED_ABLATION_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_FINRL_DIR.mkdir(parents=True, exist_ok=True)
RL_ABLATION_DIR.mkdir(parents=True, exist_ok=True)

# Healthcare innovation sector / tech sector 的原始 FinRL 长表数据
BASE_FULL_MGT_CSV = _first_existing_path(
    [
        PROJECT_ROOT / "data" / "graph" / "techtrade20241231.csv",
        PROJECT_ROOT / "data" / "techdata" / "tech" / "techtrade20241231.csv",
    ],
    "BASE_FULL_MGT_CSV"
)

# close matrix 仅在没有 *_pred_raw.csv、需要从归一化 *_pred.csv 反归一化时使用
TECH_CLOSE_CSV = _first_existing_path(
    [
        PROJECT_ROOT / "data" / "techdata" / "tech" / "tech" / "techclose20221231.csv",
        PROJECT_ROOT / "data" / "tech" / "techclose.csv",
    ],
    "TECH_CLOSE_CSV"
)

# 与 bank 组 A0--A9 编号保持一致
ALL_VARIANTS = [
    "A0_Full_MGT",
    "A1_Transformer_only",
    "A2_MGNN_only",
    "A3_SH_only",
    "A4_QA_only",
    "A5_KD_only",
    "A6_SH_QA",
    "A7_SH_KD",
    "A8_QA_KD",
    "A9_Fixed_fusion",
]

DISPLAY_NAMES = {
    "A0_Full_MGT": "A0 Full MGT",
    "A1_Transformer_only": "A1 Transformer-only",
    "A2_MGNN_only": "A2 MGNN-only",
    "A3_SH_only": "A3 SH-only",
    "A4_QA_only": "A4 QA-only",
    "A5_KD_only": "A5 KD-only",
    "A6_SH_QA": "A6 SH+QA",
    "A7_SH_KD": "A7 SH+KD",
    "A8_QA_KD": "A8 QA+KD",
    "A9_Fixed_fusion": "A9 Fixed-fusion",
}

print("\n========== PATH CHECK ==========")
print("PRED_ABLATION_DIR =", PRED_ABLATION_DIR, "| exists=", PRED_ABLATION_DIR.exists())
print("GENERATED_FINRL_DIR =", GENERATED_FINRL_DIR)
print("RL_ABLATION_DIR =", RL_ABLATION_DIR)
print("BASE_FULL_MGT_CSV =", BASE_FULL_MGT_CSV, "| exists=", BASE_FULL_MGT_CSV.exists())
print("TECH_CLOSE_CSV =", TECH_CLOSE_CSV, "| exists=", TECH_CLOSE_CSV.exists())


FeatureEngineer import: finrl.preprocessing.preprocessors
data_split import: custom fallback data_split
DRLAgent import: finrl.model.models
[OK] BASE_FULL_MGT_CSV: D:\paper\onging\data\graph\techtrade20241231.csv
[OK] TECH_CLOSE_CSV: D:\paper\onging\data\techdata\tech\tech\techclose20221231.csv

========== PATH CHECK ==========
PRED_ABLATION_DIR = D:\paper\onging\ablation_tech | exists= True
GENERATED_FINRL_DIR = D:\paper\onging\ablation_tech\finrl_csv
RL_ABLATION_DIR = D:\paper\onging\ablation_tech\rl
BASE_FULL_MGT_CSV = D:\paper\onging\data\graph\techtrade20241231.csv | exists= True
TECH_CLOSE_CSV = D:\paper\onging\data\techdata\tech\tech\techclose20221231.csv | exists= True


## 1. 与预测消融 notebook 对应的科技股输入文件配置

下面的 `RL_ABLATION_INPUTS` 与第一个文件中的 A0--A9 消融编号一一对应。

程序会优先读取自动生成的科技股 A0--A9 FinRL 长表 CSV：

```text
D:/paper/onging/ablation/finrl_csv/techtrade20241231_A0_Full_MGT.csv
D:/paper/onging/ablation/finrl_csv/techtrade20241231_A1_Transformer_only.csv
...
```

如果某个版本没有生成，程序会自动跳过；如果 A0 没生成，则回退到原始科技股数据：

```text
D:/paper/onging/data/graph/techtrade20241231.csv
```


In [2]:
# ===============================
# 1. Ablation input mapping
# ===============================

# 每个版本对应的自动生成 FinRL 长表 CSV。
# 这些 CSV 会由 D:/paper/onging/ablation_tech/A*_seed*_pred_raw.csv 自动生成。
def generated_finrl_csv_path(tag: str) -> Path:
    return GENERATED_FINRL_DIR / f"techtrade20241231_{tag}.csv"


def resolve_variant_csv(tag: str) -> Path:
    """优先使用自动生成的 A0-A9 长表 CSV；若 A0 没有生成，则回退到原始 Full MGT 文件。"""
    gen_path = generated_finrl_csv_path(tag)
    if gen_path.exists():
        return gen_path
    if tag == "A0_Full_MGT" and BASE_FULL_MGT_CSV.exists():
        return BASE_FULL_MGT_CSV
    return gen_path


RL_ABLATION_INPUTS = {tag: resolve_variant_csv(tag) for tag in ALL_VARIANTS}

print("Initial ablation CSV mapping:")
for tag, path in RL_ABLATION_INPUTS.items():
    print(f"{tag:28s} -> {path} | exists={Path(path).exists()}")


Initial ablation CSV mapping:
A0_Full_MGT                  -> D:\paper\onging\ablation_tech\finrl_csv\techtrade20241231_A0_Full_MGT.csv | exists=True
A1_Transformer_only          -> D:\paper\onging\ablation_tech\finrl_csv\techtrade20241231_A1_Transformer_only.csv | exists=True
A2_MGNN_only                 -> D:\paper\onging\ablation_tech\finrl_csv\techtrade20241231_A2_MGNN_only.csv | exists=True
A3_SH_only                   -> D:\paper\onging\ablation_tech\finrl_csv\techtrade20241231_A3_SH_only.csv | exists=True
A4_QA_only                   -> D:\paper\onging\ablation_tech\finrl_csv\techtrade20241231_A4_QA_only.csv | exists=True
A5_KD_only                   -> D:\paper\onging\ablation_tech\finrl_csv\techtrade20241231_A5_KD_only.csv | exists=True
A6_SH_QA                     -> D:\paper\onging\ablation_tech\finrl_csv\techtrade20241231_A6_SH_QA.csv | exists=True
A7_SH_KD                     -> D:\paper\onging\ablation_tech\finrl_csv\techtrade20241231_A7_SH_KD.csv | exists=True
A8_QA_KD  

## 2. 实验设置

主文建议采用：

- 初始资金：`1,000,000`
- 交易成本：`0.0001`，即万分之一
- 交易区间：`2023-01-03` 到 `2024-12-31`
- 消融实验固定一个 RL agent，例如 `A2C`
- 完整策略比较再跑 PPO/A2C/SAC/TD3/DDPG

为了先跑通代码，默认只用 `SEEDS = [1]`。论文最终结果建议改成 `SEEDS = [1, 2, 3]`。

In [3]:
# ===============================
# 2. Experiment settings
# ===============================

# 训练开始日期固定；训练结束、交易开始和交易结束默认由 pred_raw 文件自动识别。
TRAIN_START = "2018-01-02"
TRAIN_END = None       # 自动设为 TRADE_START 的前一天；也可以手动写 "2021-12-31" 或 "2022-12-31"
TRADE_START = None     # 自动由 *_pred_raw.csv 的 date 列识别
TRADE_END = None       # 自动由 *_pred_raw.csv 的 date 列识别
AUTO_DETECT_TRADE_RANGE_FROM_PRED = True

INITIAL_AMOUNT = 1_000_000
TRANSACTION_COST_PCT = 0.0001
REWARD_SCALING = 1e-1
LOOKBACK = 252

TECH_INDICATOR_LIST = ["macd", "rsi_30", "cci_30", "dx_30"]

# 与 bank 组保持一致的输出：默认先跑客观基准，并可选择是否跑五个 RL agents。
RUN_OBJECTIVE_BASELINES = True
RUN_RL_AGENTS = True   # 如果只想快速生成和 bank 当前汇总类似的基准表，可改成 False
AGENTS_FOR_ABLATION = ["a2c", "ppo", "sac", "td3", "ddpg"]

# 调试阶段用 [1]；论文最终如果需要多次重复，可改成 [1, 2, 3]
SEEDS = [1]

# 自动从 D:/paper/onging/ablation_tech/A*_seed*_pred_raw.csv 生成 A0-A9 FinRL 长表 CSV
AUTO_GENERATE_VARIANT_FINRL_CSVS = True

# 关键修正：优先使用反归一化后的 pred_raw 文件；若不存在，再回退到 pred.csv 并用 close matrix 反归一化。
PRED_FILE_KIND = "auto"  # "auto", "pred_raw", or "pred"

# 多个预测 seed 如何合并："mean" 表示对 seed1/2/3 预测取平均；也可以改成 1、2、3。
PRED_SEED_MODE = "mean"

VARIANTS_TO_RUN = ALL_VARIANTS.copy()

# 训练步数。调试时可以适当降低；正式实验建议保持和 bank 组一致。
TOTAL_TIMESTEPS = {
    "a2c": 40000,
    "ppo": 40000,
    "sac": 60000,
    "td3": 30000,
    "ddpg": 50000,
}

MODEL_PARAMS = {
    "a2c": {"n_steps": 10, "ent_coef": 0.005, "learning_rate": 0.0004},
    "ppo": {"n_steps": 2048, "ent_coef": 0.005, "learning_rate": 0.001, "batch_size": 128},
    "sac": {"batch_size": 128, "buffer_size": 1_000_000, "learning_rate": 0.0001,
            "learning_starts": 100, "ent_coef": "auto_0.1"},
    "td3": {"batch_size": 100, "buffer_size": 1_000_000, "learning_rate": 0.001},
    "ddpg": {},
}


def set_global_seed(seed: int = 1):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)


set_global_seed(1)

print("RUN_OBJECTIVE_BASELINES =", RUN_OBJECTIVE_BASELINES)
print("RUN_RL_AGENTS =", RUN_RL_AGENTS)
print("PRED_FILE_KIND =", PRED_FILE_KIND)
print("PRED_SEED_MODE =", PRED_SEED_MODE)


RUN_OBJECTIVE_BASELINES = True
RUN_RL_AGENTS = True
PRED_FILE_KIND = auto
PRED_SEED_MODE = mean


## 3. 数据读取与协方差状态构造

这部分替代原 notebook 中重复的 `FeatureEngineer` 和重复的 covariance 代码。

修正点：

1. 不再重复调用 `fe.preprocess_data(df)`；
2. 如果技术指标已经存在，则不重复生成；
3. 每次重新计算 `cov_list`，避免从 CSV 读取后变成字符串；
4. 训练集和交易集都会重新设置整数 day index，保证环境中的 `df.loc[self.day, :]` 正常工作。

In [4]:
# ===============================
# 3. Data preprocessing
# ===============================

def _reset_day_index(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["date", "tic"]).copy()
    df.index = pd.factorize(df["date"])[0]
    return df


def _check_required_columns(df: pd.DataFrame, csv_path: Path):
    required = {"date", "tic", "close"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{csv_path} 缺少必要字段: {missing}")


def load_finrl_data(csv_path, lookback=252, tech_indicator_list=None):
    """
    读取 FinRL 长表 CSV，并生成技术指标和 rolling covariance state。

    输入 CSV 至少应包含:
    date, tic, open, high, low, close, volume
    """
    csv_path = Path(csv_path)
    if not csv_path.exists():
        raise FileNotFoundError(f"找不到数据文件: {csv_path}")

    df = pd.read_csv(csv_path)
    _check_required_columns(df, csv_path)

    df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
    df["tic"] = df["tic"].astype(str)
    df = df.sort_values(["date", "tic"]).reset_index(drop=True)

    if tech_indicator_list is None:
        tech_indicator_list = TECH_INDICATOR_LIST

    missing_tech = [c for c in tech_indicator_list if c not in df.columns]
    if missing_tech:
        print(f"[{csv_path.name}] Missing technical indicators {missing_tech}; running FeatureEngineer...")
        fe = FeatureEngineer(
            use_technical_indicator=True,
            use_turbulence=False,
            user_defined_feature=False
        )
        df = fe.preprocess_data(df)
        df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
        df["tic"] = df["tic"].astype(str)
    else:
        print(f"[{csv_path.name}] Technical indicators already exist; skip FeatureEngineer.")

    # 避免从 CSV 读取旧 cov_list/return_list 后变成字符串
    for col in ["cov_list", "return_list"]:
        if col in df.columns:
            df = df.drop(columns=[col])

    df = df.sort_values(["date", "tic"], ignore_index=True)
    df.index = pd.factorize(df["date"])[0]

    unique_dates = df["date"].unique()
    if len(unique_dates) <= lookback:
        raise ValueError(f"数据交易日数量 {len(unique_dates)} 小于等于 lookback={lookback}，无法构造协方差状态。")

    cov_list = []
    return_list = []

    for i in range(lookback, len(unique_dates)):
        data_lookback = df.loc[i - lookback:i, :]
        price_lookback = data_lookback.pivot_table(index="date", columns="tic", values="close")
        return_lookback = price_lookback.pct_change().dropna()

        return_list.append(return_lookback)
        cov_list.append(return_lookback.cov().values)

    df_cov = pd.DataFrame({
        "date": unique_dates[lookback:],
        "cov_list": cov_list,
        "return_list": return_list
    })

    df = df.reset_index(drop=True)
    df = df.merge(df_cov, on="date", how="inner")
    df = df.sort_values(["date", "tic"]).reset_index(drop=True)

    return df


def _slice_by_date_inclusive(df: pd.DataFrame, start: str, end: str) -> pd.DataFrame:
    if start is None or end is None:
        raise ValueError("TRAIN/TRADE 日期尚未设置。请先运行自动读取 pred_raw 的 cell。")
    out = df[(df["date"] >= start) & (df["date"] <= end)].copy()
    out = out.sort_values(["date", "tic"]).copy()
    out.index = pd.factorize(out["date"])[0]
    return out


def make_train_trade(df: pd.DataFrame):
    """使用显式日期切片，避免 FinRL data_split 的开闭区间差异导致少一天。"""
    train = _slice_by_date_inclusive(df, TRAIN_START, TRAIN_END)
    trade = _slice_by_date_inclusive(df, TRADE_START, TRADE_END)

    train = _reset_day_index(train)
    trade = _reset_day_index(trade)

    print(f"train dates: {train['date'].min()} -> {train['date'].max()} | rows={len(train)}")
    print(f"trade dates: {trade['date'].min()} -> {trade['date'].max()} | rows={len(trade)}")

    return train, trade


## 3.1 从 `D:/paper/onging/ablation_tech` 自动读取 A0--A9 预测结果

本单元是本次修正的核心。

原代码读取不到数据的主要原因通常是：

1. 读取目录仍然是 `D:/paper/onging/ablation`，而你的 Healthcare innovation sector 预测结果实际在 `D:/paper/onging/ablation_tech`；
2. 原代码查找 `*_pred.csv`，但你的可用于回测的真实价格预测文件是 `*_pred_raw.csv`；
3. 预测文件中第一列是 `date`，原代码直接 `astype(float)` 会把日期字符串也转成数值，导致报错；
4. 原代码手动指定 2023--2024 交易区间，而当前预测文件可能是 2022 测试集，导致行数和日期数量不一致。

下面代码会自动处理这些问题。


In [5]:
# ===============================
# 3.1 Auto-generate FinRL CSVs from prediction ablation outputs
# ===============================

from sklearn.preprocessing import MinMaxScaler


def load_close_matrix_and_scaler(required_columns=None):
    """读取 close 矩阵并复现 MinMaxScaler(feature_range=(-1, 1)).

    仅当没有 *_pred_raw.csv、需要从归一化 *_pred.csv 反归一化时使用。
    """
    if TECH_CLOSE_CSV.exists():
        close = pd.read_csv(TECH_CLOSE_CSV, index_col=0)
        source = TECH_CLOSE_CSV
    else:
        if not BASE_FULL_MGT_CSV.exists():
            raise FileNotFoundError(
                f"既找不到 TECH_CLOSE_CSV: {TECH_CLOSE_CSV}，也找不到 BASE_FULL_MGT_CSV: {BASE_FULL_MGT_CSV}"
            )
        base = pd.read_csv(BASE_FULL_MGT_CSV)
        required = {"date", "tic", "close"}
        missing = required - set(base.columns)
        if missing:
            raise ValueError(f"{BASE_FULL_MGT_CSV} 缺少用于生成 close 矩阵的字段: {missing}")
        base["date"] = pd.to_datetime(base["date"]).dt.strftime("%Y-%m-%d")
        base["tic"] = base["tic"].astype(str)
        close = base.pivot_table(index="date", columns="tic", values="close", aggfunc="last")
        source = BASE_FULL_MGT_CSV
        print(f"[INFO] 未找到 {TECH_CLOSE_CSV.name}，已从 {BASE_FULL_MGT_CSV.name} 自动生成 close matrix。")

    close.index = pd.to_datetime(close.index).strftime("%Y-%m-%d")
    close.columns = close.columns.astype(str)
    close = close.sort_index()
    close = close.apply(pd.to_numeric, errors="coerce").dropna(how="all").ffill().bfill()

    if required_columns is not None:
        required_columns = [str(c) for c in required_columns]
        missing_cols = [c for c in required_columns if c not in close.columns]
        if missing_cols:
            raise ValueError(f"close matrix 缺少预测文件中的股票列: {missing_cols[:10]}")
        close = close[required_columns]

    scaler = MinMaxScaler(feature_range=(-1, 1))
    scaler.fit(close.values)
    print(f"Close matrix source = {source}; shape = {close.shape}")
    return close, scaler


def list_prediction_files():
    """列出 ablation_tech 目录中可用的 A0--A9 预测文件，便于检查命名。"""
    rows = []
    for tag in ALL_VARIANTS:
        raw_files = sorted(PRED_ABLATION_DIR.glob(f"{tag}_seed*_pred_raw.csv"))
        scaled_files = sorted(PRED_ABLATION_DIR.glob(f"{tag}_seed*_pred.csv"))
        true_raw_files = sorted(PRED_ABLATION_DIR.glob(f"{tag}_seed*_true_raw.csv"))
        rows.append({
            "variant": tag,
            "pred_raw_count": len(raw_files),
            "pred_count": len(scaled_files),
            "true_raw_count": len(true_raw_files),
            "pred_raw_files": ", ".join([p.name for p in raw_files[:3]]),
        })
    out = pd.DataFrame(rows)
    display(out)
    return out


prediction_file_check = list_prediction_files()


def _drop_unnamed_columns(df: pd.DataFrame) -> pd.DataFrame:
    unnamed = [c for c in df.columns if str(c).lower().startswith("unnamed")]
    if unnamed:
        df = df.drop(columns=unnamed)
    return df


def _detect_date_column(df: pd.DataFrame):
    for c in df.columns:
        if str(c).strip().lower() in {"date", "datetime", "trade_date", "time"}:
            return c
    return None


def find_prediction_files(tag: str):
    """按图片命名规则查找预测文件。优先 *_pred_raw.csv。"""
    if PRED_FILE_KIND in ["auto", "pred_raw"]:
        raw_files = sorted(PRED_ABLATION_DIR.glob(f"{tag}_seed*_pred_raw.csv"))
        if raw_files:
            return raw_files, "pred_raw"
        if PRED_FILE_KIND == "pred_raw":
            return [], "pred_raw"

    scaled_files = sorted(PRED_ABLATION_DIR.glob(f"{tag}_seed*_pred.csv"))
    return scaled_files, "pred"


def read_single_prediction_file(file_path: Path, file_kind: str):
    """读取一个 prediction matrix，返回 wide DataFrame(index=date if available, columns=tickers)."""
    df = pd.read_csv(file_path)
    df = _drop_unnamed_columns(df)

    date_col = _detect_date_column(df)
    if date_col is not None:
        dates = pd.to_datetime(df[date_col]).dt.strftime("%Y-%m-%d")
        values = df.drop(columns=[date_col]).copy()
    else:
        dates = None
        values = df.copy()

    values.columns = values.columns.astype(str)
    values = values.apply(pd.to_numeric, errors="coerce")
    if values.isna().any().any():
        bad_cols = values.columns[values.isna().any()].tolist()[:10]
        raise ValueError(f"{file_path.name} 中存在无法转成数值的预测列: {bad_cols}")

    if file_kind == "pred":
        # 归一化预测值需要反归一化
        close_df, scaler = load_close_matrix_and_scaler(required_columns=values.columns.tolist())
        real_values = scaler.inverse_transform(values.values)
        values = pd.DataFrame(real_values, columns=values.columns)

    if dates is not None:
        values.index = dates
        values.index.name = "date"

    return values


def _select_files_by_seed(files, seed_mode):
    if seed_mode == "mean":
        return files
    seed = int(seed_mode)
    selected = [p for p in files if f"_seed{seed}_" in p.name]
    if not selected:
        raise FileNotFoundError(f"找不到 seed={seed} 对应预测文件。候选文件: {[p.name for p in files]}")
    return selected


def read_prediction_wide(tag: str, seed_mode="mean"):
    """读取某个 A0--A9 版本的预测矩阵。多个 seed 默认取平均。"""
    files, file_kind = find_prediction_files(tag)
    if not files:
        raise FileNotFoundError(
            f"找不到 {tag}_seed*_pred_raw.csv 或 {tag}_seed*_pred.csv。\n"
            f"当前读取目录: {PRED_ABLATION_DIR}\n"
            "请确认文件命名是否类似图片中的 A0_Full_MGT_seed1_pred_raw.csv。"
        )

    files = _select_files_by_seed(files, seed_mode)
    mats = []
    for f in files:
        mat = read_single_prediction_file(f, file_kind=file_kind)
        mats.append(mat)

    # 检查列一致
    columns0 = mats[0].columns.tolist()
    for mat, f in zip(mats, files):
        if mat.columns.tolist() != columns0:
            raise ValueError(f"{tag}: {f.name} 的股票列与第一个 seed 不一致。")

    # 如果文件带 date 列，要求 date 完全一致；如果没有 date，则后续再由 close matrix 推断。
    has_index_dates = mats[0].index.name == "date"
    if has_index_dates:
        index0 = mats[0].index.tolist()
        for mat, f in zip(mats, files):
            if mat.index.tolist() != index0:
                raise ValueError(f"{tag}: {f.name} 的 date 行与第一个 seed 不一致。")
        mean_values = np.mean(np.stack([m.values for m in mats], axis=0), axis=0)
        pred_wide = pd.DataFrame(mean_values, index=mats[0].index, columns=columns0)
        pred_wide.index.name = "date"
    else:
        mean_values = np.mean(np.stack([m.values for m in mats], axis=0), axis=0)
        pred_wide = pd.DataFrame(mean_values, columns=columns0)

    print(f"[READ] {tag}: kind={file_kind}, seed_mode={seed_mode}, files={[p.name for p in files]}")
    print(f"       pred_wide shape={pred_wide.shape}, date_index={pred_wide.index.name == 'date'}")
    return pred_wide, files, file_kind


def prediction_dates_from_close(close_df: pd.DataFrame, n_rows: int, start_date: str, end_date: str):
    dates = pd.Series(close_df.index, name="date")
    mask = (dates >= start_date) & (dates <= end_date)
    selected = dates.loc[mask].tolist()

    if len(selected) != n_rows:
        raise ValueError(
            "预测矩阵行数与指定日期区间不一致。\n"
            f"  pred rows     = {n_rows}\n"
            f"  selected days = {len(selected)}\n"
            f"  date range    = {start_date} ~ {end_date}\n"
            "解决方法：优先使用带 date 列的 *_pred_raw.csv，或手动调整 TRADE_START/TRADE_END。"
        )
    return selected


def _infer_trade_range_from_generated_csvs(generated_paths):
    starts, ends = [], []
    for tag, path in generated_paths.items():
        if not Path(path).exists():
            continue
        # 直接从对应 pred_raw 读取日期更准确
        try:
            pred_wide, _, _ = read_prediction_wide(tag, seed_mode=PRED_SEED_MODE)
            if pred_wide.index.name == "date":
                starts.append(min(pred_wide.index))
                ends.append(max(pred_wide.index))
        except Exception:
            pass
    if not starts:
        return None, None
    return min(starts), max(ends)


def build_variant_finrl_csv_from_pred(tag: str, seed_mode="mean", start_date=None, end_date=None):
    """把 A*_seed*_pred_raw.csv 转成 FinRL 长表 CSV。"""
    pred_wide, used_files, file_kind = read_prediction_wide(tag, seed_mode=seed_mode)

    if pred_wide.index.name != "date":
        # 只有不带 date 列时才需要用 close matrix 和手工日期区间推断
        close_df, _ = load_close_matrix_and_scaler(required_columns=pred_wide.columns.tolist())
        if start_date is None or end_date is None:
            raise ValueError(f"{tag}: 预测文件没有 date 列，必须手动设置 TRADE_START/TRADE_END。")
        dates = prediction_dates_from_close(close_df, pred_wide.shape[0], start_date, end_date)
        pred_wide.index = dates
        pred_wide.index.name = "date"

    pred_wide = pred_wide.copy()
    pred_wide.index = pd.to_datetime(pred_wide.index).strftime("%Y-%m-%d")
    pred_wide.columns = pred_wide.columns.astype(str)

    pred_long = pred_wide.reset_index().melt(id_vars="date", var_name="tic", value_name="pred_close")
    pred_long["tic"] = pred_long["tic"].astype(str)

    if not BASE_FULL_MGT_CSV.exists():
        raise FileNotFoundError(f"找不到原始 FinRL 长表数据: {BASE_FULL_MGT_CSV}")

    base = pd.read_csv(BASE_FULL_MGT_CSV)
    _check_required_columns(base, BASE_FULL_MGT_CSV)
    base["date"] = pd.to_datetime(base["date"]).dt.strftime("%Y-%m-%d")
    base["tic"] = base["tic"].astype(str)

    # ticker 对齐检查
    base_tickers = set(base["tic"].unique())
    pred_tickers = set(pred_wide.columns)
    missing_in_base = sorted(pred_tickers - base_tickers)
    if missing_in_base:
        raise ValueError(
            f"{tag}: 预测文件 ticker 与 FinRL 长表 ticker 不一致。\n"
            f"预测中有但 base 中没有: {missing_in_base[:20]}"
        )

    # 只替换预测区间内的 close，其余日期保留真实 close，用于训练和构造协方差状态。
    merged = base.merge(pred_long, on=["date", "tic"], how="left")
    replace_mask = merged["pred_close"].notna()
    merged.loc[replace_mask, "close"] = merged.loc[replace_mask, "pred_close"]
    merged = merged.drop(columns=["pred_close"])

    out_csv = generated_finrl_csv_path(tag)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(out_csv, index=False)

    print(f"[OK] {tag}: generated {out_csv}")
    print(f"     kind={file_kind}; used files: {[p.name for p in used_files]}")
    print(f"     replaced close rows: {int(replace_mask.sum())}")
    print(f"     prediction date range: {pred_wide.index.min()} -> {pred_wide.index.max()}")
    return out_csv


def auto_generate_all_variant_finrl_csvs():
    generated = {}
    for tag in ALL_VARIANTS:
        try:
            generated[tag] = build_variant_finrl_csv_from_pred(
                tag=tag,
                seed_mode=PRED_SEED_MODE,
                start_date=TRADE_START,
                end_date=TRADE_END,
            )
        except Exception as e:
            print(f"[WARN] {tag}: cannot generate FinRL CSV -> {e}")
    return generated


if AUTO_GENERATE_VARIANT_FINRL_CSVS:
    print("\nStart auto-generating FinRL CSVs from prediction ablation outputs...")
    generated_paths = auto_generate_all_variant_finrl_csvs()
else:
    generated_paths = {}
    print("AUTO_GENERATE_VARIANT_FINRL_CSVS=False; skip auto generation.")

# 自动识别交易区间，使回测区间与 pred_raw 文件完全一致。
if AUTO_DETECT_TRADE_RANGE_FROM_PRED:
    inferred_start, inferred_end = _infer_trade_range_from_generated_csvs(generated_paths)
    if inferred_start is not None:
        TRADE_START = inferred_start
        TRADE_END = inferred_end
        if TRAIN_END is None:
            TRAIN_END = (pd.to_datetime(TRADE_START) - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
        print("\n[DATE AUTO-DETECT]")
        print("TRAIN_START =", TRAIN_START)
        print("TRAIN_END   =", TRAIN_END)
        print("TRADE_START =", TRADE_START)
        print("TRADE_END   =", TRADE_END)
    else:
        raise ValueError("无法从预测文件自动识别交易区间。请检查 *_pred_raw.csv 是否包含 date 列。")
else:
    if TRAIN_END is None:
        raise ValueError("AUTO_DETECT_TRADE_RANGE_FROM_PRED=False 时必须手动设置 TRAIN_END。")

# 刷新映射：优先使用刚生成的 finrl_csv；A0 如果没生成，则回退原始 BASE_FULL_MGT_CSV
RL_ABLATION_INPUTS = {tag: resolve_variant_csv(tag) for tag in ALL_VARIANTS}

print("\nRefreshed ablation CSV mapping:")
for tag, path in RL_ABLATION_INPUTS.items():
    print(f"{tag:28s} -> {path} | exists={Path(path).exists()}")


,variant,pred_raw_count,pred_count,true_raw_count,pred_raw_files
0,A0_Full_MGT,3,3,3,"A0_Full_MGT_seed1_pred_raw.csv, A0_Full_MGT_se..."
1,A1_Transformer_only,3,3,3,"A1_Transformer_only_seed1_pred_raw.csv, A1_Tra..."
2,A2_MGNN_only,3,3,3,"A2_MGNN_only_seed1_pred_raw.csv, A2_MGNN_only_..."
3,A3_SH_only,3,3,3,"A3_SH_only_seed1_pred_raw.csv, A3_SH_only_seed..."
4,A4_QA_only,3,3,3,"A4_QA_only_seed1_pred_raw.csv, A4_QA_only_seed..."
5,A5_KD_only,3,3,3,"A5_KD_only_seed1_pred_raw.csv, A5_KD_only_seed..."
6,A6_SH_QA,3,3,3,"A6_SH_QA_seed1_pred_raw.csv, A6_SH_QA_seed2_pr..."
7,A7_SH_KD,3,3,3,"A7_SH_KD_seed1_pred_raw.csv, A7_SH_KD_seed2_pr..."
8,A8_QA_KD,3,3,3,"A8_QA_KD_seed1_pred_raw.csv, A8_QA_KD_seed2_pr..."
9,A9_Fixed_fusion,3,3,3,"A9_Fixed_fusion_seed1_pred_raw.csv, A9_Fixed_f..."



Start auto-generating FinRL CSVs from prediction ablation outputs...
[READ] A0_Full_MGT: kind=pred_raw, seed_mode=mean, files=['A0_Full_MGT_seed1_pred_raw.csv', 'A0_Full_MGT_seed2_pred_raw.csv', 'A0_Full_MGT_seed3_pred_raw.csv']
       pred_wide shape=(242, 15), date_index=True
[OK] A0_Full_MGT: generated D:\paper\onging\ablation_tech\finrl_csv\techtrade20241231_A0_Full_MGT.csv
     kind=pred_raw; used files: ['A0_Full_MGT_seed1_pred_raw.csv', 'A0_Full_MGT_seed2_pred_raw.csv', 'A0_Full_MGT_seed3_pred_raw.csv']
     replaced close rows: 3630
     prediction date range: 2022-01-04 -> 2022-12-30
[READ] A1_Transformer_only: kind=pred_raw, seed_mode=mean, files=['A1_Transformer_only_seed1_pred_raw.csv', 'A1_Transformer_only_seed2_pred_raw.csv', 'A1_Transformer_only_seed3_pred_raw.csv']
       pred_wide shape=(242, 15), date_index=True
[OK] A1_Transformer_only: generated D:\paper\onging\ablation_tech\finrl_csv\techtrade20241231_A1_Transformer_only.csv
     kind=pred_raw; used files: ['A1_Tr

## 4. 修正版 Portfolio Allocation Environment

相对于原 notebook，主要修正：

1. `transaction_cost_pct` 不再是 0，默认使用万分之一；
2. 在 `step()` 中显式计算 turnover 和 transaction cost；
3. 保存 `turnover_memory` 和 `cost_memory`；
4. reward 使用 `portfolio_return * reward_scaling`，避免直接用账户净值导致奖励尺度过大；
5. `save_asset_memory()` 输出 daily return 和 account value。

In [6]:
# ===============================
# 4. Portfolio allocation environment with transaction cost
# ===============================

class StockPortfolioEnv(gym.Env):
    metadata = {"render.modes": ["human"]}

    def __init__(
        self,
        df,
        stock_dim,
        hmax,
        initial_amount,
        transaction_cost_pct,
        reward_scaling,
        state_space,
        action_space,
        tech_indicator_list,
        turbulence_threshold=None,
        lookback=252,
        day=0,
    ):
        self.day = day
        self.lookback = lookback
        self.df = df
        self.stock_dim = stock_dim
        self.hmax = hmax
        self.initial_amount = initial_amount
        self.transaction_cost_pct = transaction_cost_pct
        self.reward_scaling = reward_scaling
        self.state_space = state_space
        self.action_space_dim = action_space
        self.tech_indicator_list = tech_indicator_list
        self.turbulence_threshold = turbulence_threshold

        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(self.action_space_dim,), dtype=np.float32)
        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(self.state_space + len(self.tech_indicator_list), self.state_space),
            dtype=np.float32,
        )

        self.data = self.df.loc[self.day, :]
        self.covs = self.data["cov_list"].values[0]
        self.state = np.append(
            np.array(self.covs),
            [self.data[tech].values.tolist() for tech in self.tech_indicator_list],
            axis=0,
        )

        self.terminal = False
        self.portfolio_value = self.initial_amount
        self.reward = 0.0

        self.asset_memory = [self.initial_amount]
        self.portfolio_return_memory = [0.0]
        self.actions_memory = [[1 / self.stock_dim] * self.stock_dim]
        self.turnover_memory = [0.0]
        self.cost_memory = [0.0]
        self.date_memory = [self.data.date.unique()[0]]

    def step(self, actions):
        self.terminal = self.day >= len(self.df.index.unique()) - 1

        if self.terminal:
            return self.state.astype(np.float32), self.reward, self.terminal, {}

        # 当前动作 -> long-only simplex 权重
        weights = self.softmax_normalization(actions)
        prev_weights = np.array(self.actions_memory[-1], dtype=float)

        # 当前日价格记忆
        last_day_memory = self.data

        # 进入下一交易日
        self.day += 1
        self.data = self.df.loc[self.day, :]

        self.covs = self.data["cov_list"].values[0]
        self.state = np.append(
            np.array(self.covs),
            [self.data[tech].values.tolist() for tech in self.tech_indicator_list],
            axis=0,
        )

        # 组合收益、换手率和交易成本
        price_relative = self.data.close.values / last_day_memory.close.values
        gross_return = float(np.sum((price_relative - 1.0) * weights))

        turnover = float(np.sum(np.abs(weights - prev_weights)))
        cost = float(self.transaction_cost_pct * turnover)

        portfolio_return = gross_return - cost

        # 更新账户
        new_portfolio_value = self.portfolio_value * (1.0 + portfolio_return)
        self.portfolio_value = new_portfolio_value

        # 保存轨迹
        self.actions_memory.append(weights)
        self.turnover_memory.append(turnover)
        self.cost_memory.append(cost)
        self.portfolio_return_memory.append(portfolio_return)
        self.asset_memory.append(new_portfolio_value)
        self.date_memory.append(self.data.date.unique()[0])

        # 使用收益率作为 reward，避免账户值尺度过大
        self.reward = portfolio_return * self.reward_scaling

        return self.state.astype(np.float32), self.reward, self.terminal, {}

    def reset(self):
        self.asset_memory = [self.initial_amount]
        self.day = 0
        self.data = self.df.loc[self.day, :]

        self.covs = self.data["cov_list"].values[0]
        self.state = np.append(
            np.array(self.covs),
            [self.data[tech].values.tolist() for tech in self.tech_indicator_list],
            axis=0,
        )

        self.portfolio_value = self.initial_amount
        self.reward = 0.0
        self.terminal = False
        self.portfolio_return_memory = [0.0]
        self.actions_memory = [[1 / self.stock_dim] * self.stock_dim]
        self.turnover_memory = [0.0]
        self.cost_memory = [0.0]
        self.date_memory = [self.data.date.unique()[0]]

        return self.state.astype(np.float32)

    def render(self, mode="human"):
        return self.state

    def softmax_normalization(self, actions):
        # SB3/DummyVecEnv may pass actions with shape (1, N); flatten to (N,)
        actions = np.array(actions, dtype=float).reshape(-1)
        actions = actions - np.max(actions)
        numerator = np.exp(actions)
        denominator = np.sum(numerator)
        if denominator == 0 or not np.isfinite(denominator):
            return np.ones(self.stock_dim) / self.stock_dim
        return numerator / denominator

    def save_asset_memory(self):
        return pd.DataFrame({
            "date": self.date_memory,
            "daily_return": self.portfolio_return_memory,
            "account_value": self.asset_memory,
            "turnover": self.turnover_memory,
            "transaction_cost": self.cost_memory,
        })

    def save_action_memory(self):
        df_date = pd.DataFrame(self.date_memory, columns=["date"])
        df_actions = pd.DataFrame(self.actions_memory)
        df_actions.columns = self.data.tic.values
        df_actions.index = df_date.date
        return df_actions

    def _seed(self, seed=None):
        self.np_random, seed = seeding.np_random(seed)
        return [seed]

    def get_sb_env(self):
        e = DummyVecEnv([lambda: self])
        obs = e.reset()
        return e, obs


## 5. 评价指标

输出指标包括：

- CR：累计收益率
- AR：年化收益率
- Vol：年化波动率
- SR：Sharpe ratio
- MDD：最大回撤
- Calmar：Calmar ratio
- Sortino：Sortino ratio
- Omega：Omega ratio
- Turnover：平均换手率
- FinalAccountValue：期末账户价值

In [7]:
# ===============================
# 5. Metrics
# ===============================

def calculate_portfolio_metrics(df_daily_return: pd.DataFrame, df_actions: pd.DataFrame = None):
    r = df_daily_return["daily_return"].astype(float).values

    if len(r) == 0:
        raise ValueError("daily_return is empty.")

    cumulative_return = float(np.prod(1.0 + r) - 1.0)
    annual_return = float((1.0 + cumulative_return) ** (252 / max(len(r), 1)) - 1.0)
    annual_vol = float(np.std(r, ddof=1) * np.sqrt(252)) if len(r) > 1 else np.nan

    sharpe = annual_return / annual_vol if annual_vol and annual_vol != 0 else np.nan

    cumulative_curve = np.cumprod(1.0 + r)
    running_max = np.maximum.accumulate(cumulative_curve)
    drawdown = cumulative_curve / running_max - 1.0
    mdd = float(np.min(drawdown))

    calmar = annual_return / abs(mdd) if mdd != 0 else np.nan

    downside = r[r < 0]
    downside_vol = float(np.std(downside, ddof=1) * np.sqrt(252)) if len(downside) > 1 else np.nan
    sortino = annual_return / downside_vol if downside_vol and downside_vol != 0 else np.nan

    pos_sum = float(np.sum(r[r > 0]))
    neg_sum = float(abs(np.sum(r[r < 0])))
    omega = pos_sum / neg_sum if neg_sum != 0 else np.inf

    if "account_value" in df_daily_return.columns:
        final_account_value = float(df_daily_return["account_value"].iloc[-1])
    else:
        final_account_value = float(cumulative_curve[-1])

    if "turnover" in df_daily_return.columns:
        turnover = float(df_daily_return["turnover"].mean())
    elif df_actions is not None:
        turnover = float(df_actions.diff().abs().sum(axis=1).mean())
    else:
        turnover = np.nan

    return {
        "CR": cumulative_return,
        "AR": annual_return,
        "Vol": annual_vol,
        "SR": sharpe,
        "MDD": mdd,
        "Calmar": calmar,
        "Sortino": sortino,
        "Omega": omega,
        "Turnover": turnover,
        "FinalAccountValue": final_account_value,
    }


In [8]:

# ===============================
# 5.1 Normalize FinRL prediction outputs
# ===============================

def normalize_daily_return_df(df_daily_return):
    """Make returned daily_return dataframe robust across FinRL versions."""
    if isinstance(df_daily_return, list) and len(df_daily_return) > 0:
        df_daily_return = df_daily_return[0]
    df_daily_return = pd.DataFrame(df_daily_return).copy()

    # Some FinRL versions return account_value only
    if "daily_return" not in df_daily_return.columns:
        if "account_value" in df_daily_return.columns:
            df_daily_return["daily_return"] = (
                df_daily_return["account_value"].astype(float).pct_change().fillna(0.0)
            )
        else:
            raise ValueError(
                "df_daily_return must contain daily_return or account_value. "
                f"Columns={df_daily_return.columns.tolist()}"
            )

    if "account_value" not in df_daily_return.columns:
        df_daily_return["account_value"] = (
            INITIAL_AMOUNT * (1.0 + df_daily_return["daily_return"].astype(float)).cumprod()
        )

    if "date" not in df_daily_return.columns:
        df_daily_return.insert(0, "date", np.arange(len(df_daily_return)))

    return df_daily_return


def normalize_action_df(df_actions):
    """Make returned action dataframe robust across FinRL versions."""
    if isinstance(df_actions, list) and len(df_actions) > 0:
        df_actions = df_actions[0]
    df_actions = pd.DataFrame(df_actions).copy()
    return df_actions


## 6. 传统投资基准

这部分用于回应审稿人“缺少 objective investment baselines”的问题。

默认生成：

1. `EqualWeight_Rebalanced`：每日等权再平衡；
2. `BuyHold_EqualInitial`：初始等权买入并持有。

In [9]:
# ===============================
# 6. Objective investment baselines
# ===============================

def equal_weight_rebalanced_baseline(trade_df: pd.DataFrame):
    price = trade_df.pivot_table(index="date", columns="tic", values="close").sort_index()
    asset_ret = price.pct_change().fillna(0.0)
    daily_ret = asset_ret.mean(axis=1)

    df_ret = pd.DataFrame({
        "date": daily_ret.index,
        "daily_return": daily_ret.values,
    })
    df_ret["account_value"] = INITIAL_AMOUNT * (1.0 + df_ret["daily_return"]).cumprod()
    return df_ret

def buy_and_hold_equal_initial_baseline(trade_df: pd.DataFrame):
    price = trade_df.pivot_table(index="date", columns="tic", values="close").sort_index()
    rel_price = price / price.iloc[0]
    portfolio_curve = rel_price.mean(axis=1)
    daily_ret = portfolio_curve.pct_change().fillna(0.0)

    df_ret = pd.DataFrame({
        "date": daily_ret.index,
        "daily_return": daily_ret.values,
    })
    df_ret["account_value"] = INITIAL_AMOUNT * (1.0 + df_ret["daily_return"]).cumprod()
    return df_ret

def run_objective_baselines(trade_df: pd.DataFrame, tag: str):
    rows = []

    baselines = {
        "EqualWeight_Rebalanced": equal_weight_rebalanced_baseline(trade_df),
        "BuyHold_EqualInitial": buy_and_hold_equal_initial_baseline(trade_df),
    }

    for name, df_ret in baselines.items():
        out_path = RL_ABLATION_DIR / f"{tag}_{name}_daily_return.csv"
        df_ret.to_csv(out_path, index=False)

        metrics = calculate_portfolio_metrics(df_ret)
        metrics.update({
            "variant": tag,
            "agent": name,
            "seed": -1,
            "csv_path": "",
        })
        rows.append(metrics)

    return rows


## 7. 单个 RL 消融实验函数

这部分是主函数。

输入一个 FinRL 长表 CSV，训练指定 RL agent，并输出：

- daily return CSV
- action/weight CSV
- 单次实验指标字典

In [10]:
# ===============================
# 7. Run one RL experiment
# ===============================

def build_env_kwargs(train_df: pd.DataFrame):
    stock_dimension = len(train_df.tic.unique())
    state_space = stock_dimension

    env_kwargs = {
        "hmax": 100,
        "initial_amount": INITIAL_AMOUNT,
        "transaction_cost_pct": TRANSACTION_COST_PCT,
        "state_space": state_space,
        "stock_dim": stock_dimension,
        "tech_indicator_list": TECH_INDICATOR_LIST,
        "action_space": stock_dimension,
        "reward_scaling": REWARD_SCALING,
    }
    return env_kwargs

def train_and_trade_one_agent(csv_path, variant_tag, agent_name="a2c", seed=1):
    set_global_seed(seed)
    csv_path = Path(csv_path)

    print(f"\n========== {variant_tag} | {agent_name.upper()} | seed={seed} ==========")
    print("CSV:", csv_path)

    df = load_finrl_data(csv_path, lookback=LOOKBACK, tech_indicator_list=TECH_INDICATOR_LIST)
    train, trade = make_train_trade(df)

    if train.empty:
        raise ValueError(f"{variant_tag}: train 数据为空，请检查日期范围 {TRAIN_START} 到 {TRAIN_END}")
    if trade.empty:
        raise ValueError(f"{variant_tag}: trade 数据为空，请检查日期范围 {TRADE_START} 到 {TRADE_END}")

    env_kwargs = build_env_kwargs(train)

    e_train_gym = StockPortfolioEnv(df=train, **env_kwargs)
    env_train, _ = e_train_gym.get_sb_env()

    agent = DRLAgent(env=env_train)

    params = MODEL_PARAMS.get(agent_name, {}).copy()
    # Stable-Baselines3 大多数算法支持 seed 参数
    params["seed"] = seed

    model = agent.get_model(model_name=agent_name, model_kwargs=params)
    trained_model = agent.train_model(
        model=model,
        tb_log_name=f"{variant_tag}_{agent_name}_seed{seed}",
        total_timesteps=TOTAL_TIMESTEPS[agent_name],
    )

    e_trade_gym = StockPortfolioEnv(df=trade, **env_kwargs)

    df_daily_return, df_actions = DRLAgent.DRL_prediction(
        model=trained_model,
        environment=e_trade_gym,
    )

    # 重要修正：
    # 不要在 DRL_prediction 之后再调用 e_trade_gym.save_asset_memory() 覆盖结果。
    # DummyVecEnv 在 done=True 后可能自动 reset 环境，导致 memory 只剩初始值，
    # 进而出现 A2C 的 CR=0、AR=0、Turnover=0 的假结果。
    df_daily_return = normalize_daily_return_df(df_daily_return)
    df_actions = normalize_action_df(df_actions)

    metrics = calculate_portfolio_metrics(df_daily_return, df_actions)
    metrics.update({
        "variant": variant_tag,
        "agent": agent_name,
        "seed": seed,
        "csv_path": str(csv_path),
    })

    # Save outputs
    daily_path = RL_ABLATION_DIR / f"{variant_tag}_{agent_name}_seed{seed}_daily_return.csv"
    action_path = RL_ABLATION_DIR / f"{variant_tag}_{agent_name}_seed{seed}_actions.csv"

    df_daily_return.to_csv(daily_path, index=False)
    df_actions.to_csv(action_path)

    print("Saved:", daily_path)
    print("Saved:", action_path)
    print(metrics)

    return metrics


## 8. 批量运行 forecast-to-policy 消融实验

默认只运行 `VARIANTS_TO_RUN` 中存在 CSV 的版本。

如果某个 CSV 不存在，会显示 `[SKIP]`，不会报错。

In [11]:
# ===============================
# 8. Batch run RL ablation
# ===============================

all_metrics = []

for variant_tag in VARIANTS_TO_RUN:
    csv_path = RL_ABLATION_INPUTS[variant_tag]

    if not Path(csv_path).exists():
        print(f"[SKIP] {variant_tag}: file not found -> {csv_path}")
        continue

    # 1) 客观投资基准：与 bank 组当前汇总表保持一致，所有 A0--A9 都会生成。
    if RUN_OBJECTIVE_BASELINES:
        try:
            df_for_baseline = load_finrl_data(csv_path, lookback=LOOKBACK, tech_indicator_list=TECH_INDICATOR_LIST)
            _, trade_for_baseline = make_train_trade(df_for_baseline)
            baseline_rows = run_objective_baselines(trade_for_baseline, variant_tag)
            all_metrics.extend(baseline_rows)
        except Exception as e:
            print(f"[Warning] Objective baselines failed for {variant_tag}: {e}")

    # 2) RL forecast-to-policy ablation：如需和完整策略层实验一致，保持 RUN_RL_AGENTS=True。
    if RUN_RL_AGENTS:
        for agent_name in AGENTS_FOR_ABLATION:
            for seed in SEEDS:
                try:
                    metrics = train_and_trade_one_agent(
                        csv_path=csv_path,
                        variant_tag=variant_tag,
                        agent_name=agent_name,
                        seed=seed,
                    )
                    all_metrics.append(metrics)
                except Exception as e:
                    print(f"[ERROR] {variant_tag} | {agent_name} | seed={seed}: {e}")
    else:
        print(f"[INFO] RUN_RL_AGENTS=False; skip RL agents for {variant_tag}")

raw_metrics_df = pd.DataFrame(all_metrics)
raw_metrics_path = RL_ABLATION_DIR / "rl_ablation_raw_results.csv"
raw_metrics_df.to_csv(raw_metrics_path, index=False)

print("Saved raw metrics:", raw_metrics_path)
display(raw_metrics_df)


[techtrade20241231_A0_Full_MGT.csv] Missing technical indicators ['macd', 'rsi_30', 'cci_30', 'dx_30']; running FeatureEngineer...


C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  indicator_df = indicator_df.append(
C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  indicator_df = indicator_df.append(
C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  indicator_df = indicator_df.append(
C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future ve

C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  indicator_df = indicator_df.append(
C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  indicator_df = indicator_df.append(
C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  indicator_df = indicator_df.append(
C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future ve

C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  indicator_df = indicator_df.append(
C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  indicator_df = indicator_df.append(
C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  indicator_df = indicator_df.append(
C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future ve

Successfully added technical indicators
train dates: 2019-01-15 -> 2021-12-31 | rows=10815
trade dates: 2022-01-04 -> 2022-12-30 | rows=3630

========== A0_Full_MGT | A2C | seed=1 ==========
CSV: D:\paper\onging\ablation_tech\finrl_csv\techtrade20241231_A0_Full_MGT.csv
[techtrade20241231_A0_Full_MGT.csv] Missing technical indicators ['macd', 'rsi_30', 'cci_30', 'dx_30']; running FeatureEngineer...


...

C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  indicator_df = indicator_df.append(
C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  indicator_df = indicator_df.append(
C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  indicator_df = indicator_df.append(
C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\finrl\preprocessing\preprocessors.py:82: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future ve

Successfully added technical indicators
train dates: 2019-01-15 -> 2021-12-31 | rows=10815
trade dates: 2022-01-04 -> 2022-12-30 | rows=3630
{'seed': 1}
Using cpu device
Logging to tensorboard_log/ddpg\A9_Fixed_fusion_ddpg_seed1_1


C:\Users\perfe\.conda\envs\finrl-meta\lib\site-packages\stable_baselines3\common\vec_env\patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


---------------------------------
| time/              |          |
|    episodes        | 4        |
|    fps             | 10       |
|    time_elapsed    | 276      |
|    total_timesteps | 2884     |
| train/             |          |
|    actor_loss      | 0.621    |
|    critic_loss     | 0.0415   |
|    learning_rate   | 0.001    |
|    n_updates       | 2783     |
---------------------------------
---------------------------------
| time/              |          |
|    episodes        | 8        |
|    fps             | 10       |
|    time_elapsed    | 561      |
|    total_timesteps | 5768     |
| train/             |          |
|    actor_loss      | 0.539    |
|    critic_loss     | 0.000201 |
|    learning_rate   | 0.001    |
|    n_updates       | 5667     |
---------------------------------
---------------------------------
| time/              |          |
|    episodes        | 12       |
|    fps             | 10       |
|    time_elapsed    | 844      |
|    total_tim

,CR,AR,Vol,SR,MDD,Calmar,Sortino,Omega,Turnover,FinalAccountValue,variant,agent,seed,csv_path
0,0.165816,0.173230,0.130898,1.323400,-0.084681,2.045683,2.371352,1.384396,NaN,1.165816e+06,A0_Full_MGT,EqualWeight_Rebalanced,-1,
1,0.113393,0.118346,0.114187,1.036422,-0.091705,1.290509,1.635985,1.271738,NaN,1.113393e+06,A0_Full_MGT,BuyHold_EqualInitial,-1,
2,0.118282,0.123460,0.153311,0.805294,-0.123064,1.003218,1.332812,1.229469,0.735573,1.118282e+06,A0_Full_MGT,a2c,1,D:\paper\onging\ablation_tech\finrl_csv\techtr...
3,0.242173,0.253355,0.172984,1.464611,-0.101989,2.484134,3.193811,1.498058,0.752359,1.242173e+06,A0_Full_MGT,ppo,1,D:\paper\onging\ablation_tech\finrl_csv\techtr...
4,0.071126,0.074172,0.111642,0.664370,-0.094907,0.781521,0.997358,1.163796,0.679056,1.071126e+06,A0_Full_MGT,sac,1,D:\paper\onging\ablation_tech\finrl_csv\techtr...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57,0.155492,0.162413,0.203678,0.797400,-0.130795,1.241733,1.214993,1.226279,0.758141,1.155492e+06,A9_Fixed_fusion,a2c,1,D:\paper\onging\ablation_tech\finrl_csv\techtr...
58,0.232495,0.243187,0.262541,0.926284,-0.120306,2.021414,1.811160,1.305771,0.753289,1.232495e+06,A9_Fixed_fusion,ppo,1,D:\paper\onging\ablation_tech\finrl_csv\techtr...
59,0.367263,0.385051,0.285715,1.347676,-0.110827,3.474338,2.785930,1.463819,0.606764,1.367263e+06,A9_Fixed_fusion,sac,1,D:\paper\onging\ablation_tech\finrl_csv\techtr...
60,0.079705,0.083132,0.148718,0.558988,-0.116617,0.712862,0.807895,1.144866,0.006606,1.079705e+06,A9_Fixed_fusion,td3,1,D:\paper\onging\ablation_tech\finrl_csv\techtr...


## 9. 汇总结果表

生成论文中可用的 `mean ± std` 汇总表。

In [12]:
# ===============================
# 9. Summary and paper tables
# ===============================

if len(raw_metrics_df) == 0:
    print("No results. Please check prediction files, generated FinRL CSVs, and date settings.")
else:
    metric_cols = ["CR", "AR", "Vol", "SR", "MDD", "Calmar", "Sortino", "Omega", "Turnover", "FinalAccountValue"]
    agent_order = ["BuyHold_EqualInitial", "EqualWeight_Rebalanced"] + AGENTS_FOR_ABLATION

    # mean/std summary for repeated seeds
    summary_df = raw_metrics_df.groupby(["variant", "agent"], sort=False)[metric_cols].agg(["mean", "std"])
    summary_path = RL_ABLATION_DIR / "rl_ablation_summary_mean_std.csv"
    summary_df.to_csv(summary_path)

    # Paper table: one row for each variant-agent, using mean across seeds.
    paper_table = raw_metrics_df.groupby(["variant", "agent"], sort=False)[metric_cols].mean(numeric_only=True).reset_index()
    paper_table["variant"] = pd.Categorical(paper_table["variant"], categories=ALL_VARIANTS, ordered=True)
    paper_table["agent"] = pd.Categorical(paper_table["agent"], categories=agent_order, ordered=True)
    paper_table = paper_table.sort_values(["variant", "agent"]).reset_index(drop=True)
    paper_table["variant"] = paper_table["variant"].astype(str)
    paper_table["agent"] = paper_table["agent"].astype(str)

    paper_path = RL_ABLATION_DIR / "rl_ablation_paper_table.csv"
    paper_table.to_csv(paper_path, index=False)

    # Best agent by variant: choose highest FinalAccountValue / CR.
    best_agent = (
        paper_table.sort_values(["variant", "FinalAccountValue"], ascending=[True, False])
        .groupby("variant", sort=False)
        .head(1)
        .reset_index(drop=True)
    )
    best_path = RL_ABLATION_DIR / "best_agent_by_variant.csv"
    best_agent.to_csv(best_path, index=False)

    # Pivots consistent with bank group outputs
    pivot_final = paper_table.pivot(index="variant", columns="agent", values="FinalAccountValue").reset_index()
    pivot_final_path = RL_ABLATION_DIR / "pivot_final_account_value_A0_A9_agents.csv"
    pivot_final.to_csv(pivot_final_path, index=False)

    pivot_sharpe = paper_table.pivot(index="variant", columns="agent", values="SR").reset_index()
    pivot_sharpe_path = RL_ABLATION_DIR / "pivot_sharpe_A0_A9_agents.csv"
    pivot_sharpe.to_csv(pivot_sharpe_path, index=False)

    # LaTeX table for paper draft
    latex_path = RL_ABLATION_DIR / "rl_ablation_paper_table.tex"
    paper_table.to_latex(latex_path, index=False, float_format="%.4f", escape=False)

    print("Saved summary:", summary_path)
    print("Saved paper table:", paper_path)
    print("Saved best-agent table:", best_path)
    print("Saved pivot final value:", pivot_final_path)
    print("Saved pivot Sharpe:", pivot_sharpe_path)
    print("Saved LaTeX table:", latex_path)

    display(paper_table)
    print("\nBest agent by variant:")
    display(best_agent)


Saved summary: D:\paper\onging\ablation_tech\rl\rl_ablation_summary_mean_std.csv
Saved paper table: D:\paper\onging\ablation_tech\rl\rl_ablation_paper_table.csv
Saved best-agent table: D:\paper\onging\ablation_tech\rl\best_agent_by_variant.csv
Saved pivot final value: D:\paper\onging\ablation_tech\rl\pivot_final_account_value_A0_A9_agents.csv
Saved pivot Sharpe: D:\paper\onging\ablation_tech\rl\pivot_sharpe_A0_A9_agents.csv
Saved LaTeX table: D:\paper\onging\ablation_tech\rl\rl_ablation_paper_table.tex


C:\Users\perfe\AppData\Local\Temp\ipykernel_16684\2625305587.py:48: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  paper_table.to_latex(latex_path, index=False, float_format="%.4f", escape=False)


,variant,agent,CR,AR,Vol,SR,MDD,Calmar,Sortino,Omega,Turnover,FinalAccountValue
0,A0_Full_MGT,BuyHold_EqualInitial,0.113393,0.118346,0.114187,1.036422,-0.091705,1.290509,1.635985,1.271738,NaN,1.113393e+06
1,A0_Full_MGT,EqualWeight_Rebalanced,0.165816,0.173230,0.130898,1.323400,-0.084681,2.045683,2.371352,1.384396,NaN,1.165816e+06
2,A0_Full_MGT,a2c,0.118282,0.123460,0.153311,0.805294,-0.123064,1.003218,1.332812,1.229469,0.735573,1.118282e+06
3,A0_Full_MGT,ppo,0.242173,0.253355,0.172984,1.464611,-0.101989,2.484134,3.193811,1.498058,0.752359,1.242173e+06
4,A0_Full_MGT,sac,0.071126,0.074172,0.111642,0.664370,-0.094907,0.781521,0.997358,1.163796,0.679056,1.071126e+06
...,...,...,...,...,...,...,...,...,...,...,...,...
57,A9_Fixed_fusion,a2c,0.155492,0.162413,0.203678,0.797400,-0.130795,1.241733,1.214993,1.226279,0.758141,1.155492e+06
58,A9_Fixed_fusion,ppo,0.232495,0.243187,0.262541,0.926284,-0.120306,2.021414,1.811160,1.305771,0.753289,1.232495e+06
59,A9_Fixed_fusion,sac,0.367263,0.385051,0.285715,1.347676,-0.110827,3.474338,2.785930,1.463819,0.606764,1.367263e+06
60,A9_Fixed_fusion,td3,0.079705,0.083132,0.148718,0.558988,-0.116617,0.712862,0.807895,1.144866,0.006606,1.079705e+06



Best agent by variant:


,variant,agent,CR,AR,Vol,SR,MDD,Calmar,Sortino,Omega,Turnover,FinalAccountValue
0,A0_Full_MGT,ppo,0.242173,0.253355,0.172984,1.464611,-0.101989,2.484134,3.193811,1.498058,0.752359,1.242173e+06
1,A1_Transformer_only,a2c,0.085950,0.089656,0.058689,1.527646,-0.023242,3.857422,3.378075,1.576786,0.751443,1.085950e+06
2,A2_MGNN_only,ddpg,-0.949610,-0.955463,2.610183,-0.366052,-0.985570,-0.969452,-0.527194,1.028929,0.833094,5.038969e+04
3,A3_SH_only,sac,0.120173,0.125439,0.134170,0.934919,-0.059814,2.097160,1.639415,1.367033,0.660366,1.120173e+06
4,A4_QA_only,sac,0.618087,0.650587,0.252268,2.578947,-0.079778,8.154929,5.440345,1.768459,0.656022,1.618087e+06
5,A5_KD_only,sac,0.215116,0.224939,0.185797,1.210673,-0.095334,2.359475,2.880954,1.448320,0.660025,1.215116e+06
6,A6_SH_QA,a2c,6.774664,7.462266,4.238123,1.760748,-0.284943,26.188640,18.834952,3.440736,0.767260,7.774664e+06
7,A7_SH_KD,ppo,0.355487,0.372631,0.214114,1.740342,-0.099684,3.738120,3.947881,1.529203,0.750549,1.355487e+06
8,A8_QA_KD,td3,-0.608539,-0.623419,0.637174,-0.978413,-0.645616,-0.965619,-0.718787,0.499917,0.003301,3.914614e+05
9,A9_Fixed_fusion,sac,0.367263,0.385051,0.285715,1.347676,-0.110827,3.474338,2.785930,1.463819,0.606764,1.367263e+06


## 10. 画累计收益曲线

生成用于论文的 SVG 图：

```text
D:/paper/onging/ablation/rl/rl_ablation_cumulative_return.svg
```

建议放在 `Ablation study on forecast-driven portfolio allocation` 小节。

In [13]:
# ===============================
# 10. Plot cumulative return curves
# ===============================

def plot_cumulative_return_curves(results_dir=RL_ABLATION_DIR):
    results_dir = Path(results_dir)
    files = sorted(results_dir.glob("*_daily_return.csv"))

    if not files:
        print("No daily_return files found.")
        return

    plt.figure(figsize=(10, 5))

    for file in files:
        df_ret = pd.read_csv(file)
        if "daily_return" not in df_ret.columns:
            continue

        label = file.stem.replace("_daily_return", "")
        curve = (1.0 + df_ret["daily_return"].astype(float)).cumprod() - 1.0
        plt.plot(curve.values, label=label, linewidth=1.2)

    plt.xlabel("Trading day")
    plt.ylabel("Cumulative return")
    plt.legend(fontsize=6, ncol=2)
    plt.tight_layout()

    out_path = results_dir / "rl_ablation_cumulative_return.svg"
    png_path = results_dir / "rl_ablation_cumulative_return.png"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", out_path)
    print("Saved:", png_path)

plot_cumulative_return_curves()


Saved: D:\paper\onging\ablation_tech\rl\rl_ablation_cumulative_return.svg
Saved: D:\paper\onging\ablation_tech\rl\rl_ablation_cumulative_return.png


C:\Users\perfe\AppData\Local\Temp\ipykernel_16684\994057022.py:33: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 11. 可选：完整 RL agent 比较

如果你要做完整策略比较，可以把下面的 `RUN_FULL_AGENT_COMPARISON` 改成 `True`。

这部分只建议对 `A0_Full_MGT` 运行 PPO/A2C/SAC/TD3/DDPG，不建议所有消融版本都跑五个 agent，计算量太大。

In [14]:
# ===============================
# 11. Optional full-agent comparison
# ===============================

RUN_FULL_AGENT_COMPARISON = False

if RUN_FULL_AGENT_COMPARISON:
    full_agent_metrics = []
    variant_tag = "A0_Full_MGT"
    csv_path = RL_ABLATION_INPUTS[variant_tag]

    FULL_AGENTS = ["a2c", "ppo", "sac", "td3", "ddpg"]

    for agent_name in FULL_AGENTS:
        for seed in SEEDS:
            metrics = train_and_trade_one_agent(
                csv_path=csv_path,
                variant_tag=variant_tag,
                agent_name=agent_name,
                seed=seed,
            )
            full_agent_metrics.append(metrics)

    full_agent_df = pd.DataFrame(full_agent_metrics)
    full_agent_path = RL_ABLATION_DIR / "full_agent_comparison_raw_results.csv"
    full_agent_df.to_csv(full_agent_path, index=False)

    print("Saved:", full_agent_path)
    display(full_agent_df)
else:
    print("RUN_FULL_AGENT_COMPARISON = False; skip full PPO/A2C/SAC/TD3/DDPG comparison.")


RUN_FULL_AGENT_COMPARISON = False; skip full PPO/A2C/SAC/TD3/DDPG comparison.


## 12. 可选：从第一个预测消融结果生成 FinRL 长表 CSV 的辅助函数

如果第一个 notebook 输出的是 `A*_pred.csv` 预测矩阵，而不是 FinRL 长表 CSV，可以用下面函数把预测 close 合并回原始 OHLCV 长表中。

注意：

1. 预测矩阵必须已经反归一化到真实价格尺度；
2. 预测矩阵的列顺序必须与 `base_trade_csv` 中的 ticker 顺序一致，或者直接带有 ticker 列名；
3. 预测行数必须与指定日期范围内的交易日数量一致。

如果你的预测 CSV 仍然是 MinMaxScaler 后的 `[-1, 1]` 数值，必须先反归一化，否则不能用于回测。

In [15]:
# ===============================
# 12. Optional helper: merge predicted close into FinRL long table
# ===============================

def merge_predicted_close_to_finrl_csv(
    base_trade_csv,
    pred_matrix_csv,
    output_csv,
    start_date=TRADE_START,
    end_date=TRADE_END,
    tickers=None,
    pred_is_real_price=True,
):
    if not pred_is_real_price:
        raise ValueError("预测矩阵仍是归一化值，必须先反归一化到真实价格尺度。")

    base_trade_csv = Path(base_trade_csv)
    pred_matrix_csv = Path(pred_matrix_csv)
    output_csv = Path(output_csv)

    base = pd.read_csv(base_trade_csv)
    _check_required_columns(base, base_trade_csv)

    base["date"] = pd.to_datetime(base["date"]).dt.strftime("%Y-%m-%d")
    date_mask = (base["date"] >= start_date) & (base["date"] <= end_date)

    if tickers is None:
        tickers = sorted(base["tic"].unique().tolist())

    dates = sorted(base.loc[date_mask, "date"].unique().tolist())

    pred = pd.read_csv(pred_matrix_csv)

    # 如果预测 CSV 多了一列 unnamed index，则删除
    unnamed = [c for c in pred.columns if str(c).lower().startswith("unnamed")]
    if unnamed:
        pred = pred.drop(columns=unnamed)

    if pred.shape[1] == len(tickers):
        pred.columns = tickers
    else:
        missing = set(tickers) - set(pred.columns)
        if missing:
            raise ValueError(
                f"预测矩阵列数/列名与 ticker 不匹配。pred shape={pred.shape}, "
                f"num tickers={len(tickers)}, missing={missing}"
            )
        pred = pred[tickers]

    if len(pred) != len(dates):
        raise ValueError(
            f"预测矩阵行数 {len(pred)} 与日期数量 {len(dates)} 不一致。"
            f"请检查 start_date/end_date 或预测输出区间。"
        )

    pred = pred.copy()
    pred["date"] = dates
    pred_long = pred.melt(id_vars="date", var_name="tic", value_name="pred_close")

    merged = base.merge(pred_long, on=["date", "tic"], how="left")
    replace_mask = merged["pred_close"].notna()

    merged.loc[replace_mask, "close"] = merged.loc[replace_mask, "pred_close"]
    merged = merged.drop(columns=["pred_close"])

    output_csv.parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(output_csv, index=False)

    print("Saved merged FinRL CSV:", output_csv)
    print("Replaced close rows:", int(replace_mask.sum()))

    return merged

# 示例，不自动运行：
# merge_predicted_close_to_finrl_csv(
#     base_trade_csv=BASE_FULL_MGT_CSV,
#     pred_matrix_csv=ABLATION_ROOT / "A1_Transformer_only_seed1_pred_real_price.csv",
#     output_csv=GRAPH_DATA_DIR / "techtrade20241231_A1_Transformer_only.CSV",
#     start_date=TRADE_START,
#     end_date=TRADE_END,
# )


## 13. 论文中放置位置

本 notebook 产生的结果建议替换原来的 Table 7 / Table 8，放在：

```latex
\subsection{Ablation study on forecast-driven portfolio allocation}
```

Healthcare innovation sector 对应表格可命名为：

```latex
Table X. Forecast-to-policy ablation results on the healthcare innovation sector.
```

核心输出文件位于：

```text
D:/paper/onging/ablation_tech/rl/rl_ablation_paper_table.csv
D:/paper/onging/ablation_tech/rl/best_agent_by_variant.csv
D:/paper/onging/ablation_tech/rl/pivot_final_account_value_A0_A9_agents.csv
D:/paper/onging/ablation_tech/rl/pivot_sharpe_A0_A9_agents.csv
D:/paper/onging/ablation_tech/rl/rl_ablation_cumulative_return.svg
```

如果某个 A0--A9 版本被跳过，优先检查：

```text
D:/paper/onging/ablation_tech/A*_seed*_pred_raw.csv
```

是否完整生成，且文件中是否包含 `date` 列。
